# Decision Tree Regressor Example (Wine Quality Dataset)

**Goal: Predict the continuous quality score of red wine (3–8) based on its physicochemical properties.**

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

# NOTEBOOK_DIR resolves to the folder this notebook lives in.
# Jupyter sets the working directory to wherever it was launched from,
# so we use __file__ would not work — os.path.abspath('') is the correct
# approach for Jupyter notebooks.
NOTEBOOK_DIR = os.path.abspath('')

# Algorithm source files live in src/supervised/ at the repo root,
# which is four levels up from examples/supervised/<algo>/
SRC_SUP  = os.path.join(NOTEBOOK_DIR, '..', '..', '..', 'src', 'supervised')
sys.path.insert(0, SRC_SUP)

# Data files live in data/ at the repo root
DATA_DIR = os.path.join(NOTEBOOK_DIR, '..', '..', '..', 'data')
from regression_trees import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

wine = pd.read_csv(os.path.join(DATA_DIR, 'WineQT.csv')).drop(columns=['Id'])
FEATURE_COLS = [c for c in wine.columns if c != 'quality']
print(f"Dataset loaded: {wine.shape[0]} samples, {len(FEATURE_COLS)} features.")
print(f"Target — mean: {wine['quality'].mean():.2f}  std: {wine['quality'].std():.2f}")

## 2. Preprocessing

In [ ]:
X = StandardScaler().fit_transform(wine[FEATURE_COLS].values.astype(float))
y = wine['quality'].values.astype(float)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training samples: {X_tr.shape[0]}  |  Test samples: {X_te.shape[0]}")

## 3. Train

In [ ]:
dtr = DecisionTreeRegressor(max_depth=6, min_samples_leaf=5)
dtr.fit(X_tr, y_tr)
print(f'R²:  {dtr.score(X_te, y_te):.4f}')
print(f'MSE: {dtr.mse(X_te, y_te):.4f}')
print(f'Actual tree depth: {dtr.get_depth()}')

## 4. Results and Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
preds = dtr.predict(X_te)

axes[0].scatter(y_te, preds, alpha=0.4, s=18, color='teal')
lo, hi = y_te.min()-0.2, y_te.max()+0.2
axes[0].plot([lo,hi],[lo,hi],'r--',lw=1.5,label='Perfect fit')
axes[0].set_xlabel('Actual Quality'); axes[0].set_ylabel('Predicted Quality')
axes[0].set_title(f'Regression Tree - Predicted vs Actual  R²={dtr.score(X_te,y_te):.3f}', fontweight='bold')
axes[0].legend()

residuals = y_te - preds
axes[1].hist(residuals, bins=30, color='darkorchid', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', lw=1.5)
axes[1].set_xlabel('Residual'); axes[1].set_ylabel('Count')
axes[1].set_title('Regression Tree - Residual Distribution', fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Analysis

**Result: R²=0.2295, MSE=0.4288** on the continuous quality score prediction task at depth 6.

This R² (23%) is even lower than Linear Regression's 32%. The regression tree is performing worse than the linear model, which might seem counterintuitive — trees are supposed to handle non-linearity. The explanation is that the regression tree overfits in a different way: it creates piecewise constant predictions (each leaf returns the mean of a small group of wines), which introduces high variance. With only 914 training samples spread across a 6-level tree, many leaf nodes contain few wines, making their mean predictions unreliable on new data.

**The residual distribution** should be roughly bell-shaped and centred near zero. A right-skewed distribution would indicate the model consistently underpredicts high-quality wines; left-skewed would mean it overpredicts them. Given the class imbalance (most wines score 5–6), expect slight underprediction of 7–8 scores.

**The predicted vs actual scatter** will show a staircase pattern — because the tree predicts constant values in each leaf, predictions cluster at a small number of discrete values (the leaf means), while actual quality scores are integers from 3–8. This discretisation is a fundamental limitation of regression trees compared to linear models.

**MSE of 0.43** means predictions are on average about 0.65 quality points away from the true score (√0.43 ≈ 0.65).

**Key takeaway:** Single regression trees underperform linear regression on this dataset because the depth-6 constraint leaves too many samples per leaf (preventing fine discrimination) while still introducing enough variance to hurt generalisation. The ensemble methods — particularly Gradient Boosting — address this by combining many shallow trees.